# Pipeline
This notebook creates a CI/CD pipeline that runs the full MLOps system. Due to some limitations with resource limitability in AWS as well as permissions, this pipeline will take a pretrained model in instead of training as part of the pipeline. Since the problem is a computer vision problem, the training takes much longer and uses too many AWS resources. In order to adjust for this, the model is trained within a CI/CD action in Github. When changes are made to the model, or a new model is tested, the pipeline will run and create an h5 artifact and notify the maintainers that a new artifact exists. This h5 file is uploaded to an S3 bucket (outside of this notebook) and then used in the remaining processing steps. Stages in the pipeline are:
1. Determine Metrics

In [1]:
!pip install -U sagemaker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 105.3 MB/s eta 0:00:00
  Attempting uninstall: mock
    Found existing installation: mock 5.1.0
    Uninstalling mock-5.1.0:
      Successfully uninstalled mock-5.1.0
  Attempting uninstall: botocore
    Found existing installation: botocore 1.34.162
    Uninstalling botocore-1.34.162:
      Successfully uninstalled botocore-1.34.162
  Attempting uninstall: s3transfer
    Found existing installation: s3transfer 0.10.4
    Uninstalling s3transfer-0.10.4:
      Successfully uninstalled s3transfer-0.10.4
  Attempting uninstall: boto3
    Found existing installation: boto3 1.34.162
    Uninstalling boto3-1.34.162:
      Successfully uninstalled boto3-1.34.162
  Attempting uninstall: sagemaker
    Found existing installation: sagemaker 2.227.0
    Uninstalling sagemaker-2.227.0:
      Successfully uninstalled sagemaker-2.227.0
ERROR: pip's dependency resolver

In [3]:
# Setup steps
import sys

import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

sagemaker_session = sagemaker.session.Session()
region = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()
pipeline_session = PipelineSession()
default_bucket = sagemaker_session.default_bucket()
model_package_group_name = f"FERModelGroupName"

/opt/conda/lib/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [4]:
!mkdir -p code

### Define Parameters for Pipeline Execution
Define the Pipeline parameters that are used for this pipeline. This enables custom pipeline executions without having to modify the pipeline definition. The parameters defined in this workflow are:
- `processing_instance_count` - The instance count of the processing job
- `instance_type` - The `ml.*` instance type of the training job
- `model_approval_status` - The approval status to register with the trained model for CI/CD purposes (Defaults to "PendingManualApproval")
- `accuracy_threshold` - The accuracy threshold used to verify the accuracy of the model, initially set to >50%
- `s3_bucket` - The name of the s3 bucket that holds the data
- `s3_test_data_prefix` - Prefix of the directory that holds the test data (used in evaluation)
- `s3_model_prefix` - Where to find the model in s3

In [5]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
accuracy_threshold = ParameterFloat(name="AccuracyThreshold", default_value=0.5)
s3_bucket = ParameterString(name="S3Bucket", default_value="sagemaker-us-east-1-399018723364")
s3_test_data_prefix = ParameterString(name="S3TestDataPrefix", default_value="test/test/")
s3_model_prefix = ParameterString(name="S3ModelPrefix", default_value="group-5/models/")


### Define Model Evaluation Step 
This step evaluates the pre-trained model. This is a custom evaluation script that will preform the model evaluation. After the pipeline runs, the resulting `evaluation.json` file will be available for further analysis. 

In [8]:
!pip install -U tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 MB 45.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 136.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 112.5 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.1
    Uninstalling tensorboard-2.17.1:
      Successfully uninstalled tensorboard-2.17.1
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.17.0
    Uninstalling tensorflow-2.17.0:
      Successfully uninstalled tensorflow-2.17.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.1.1 requires nvidia-ml-py3==7.352.0, which is not installed.
autogluon-multimodal 1.1.1 requires jsonschema<4.22,>=4.18, but you 

In [19]:
import json
import pathlib
import tarfile
import numpy as np
import tensorflow as tf
import pickle
# from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import accuracy_score

if __name__ == "__main__":
    # Extract model if it's tarred
    model_path = "/opt/ml/processing/model/model.tar.gz"
    model_pathing = "AAI_540-Group-Project/fer_best_model.h5"

    with tarfile.open(model_path) as tar:
        tar.extractall(path=".")

    # extracted_model_path = "/opt/ml/processing/model"

    # Load model
    model = tf.keras.models.load_model(model_pathing)
    # model = pickle.load(open("cnn-model", "rb"))

    # Load test images
    test_dir = "AAI_540-Group-Project/fer2013/test"
    batch_size = 32
    img_size = (48, 48)

    test_dataset = image_dataset_from_directory(
        test_dir,
        shuffle=False,
        image_size=img_size,
        batch_size=batch_size
    )

    # Extract labels and images
    y_true = np.concatenate([y.numpy() for _, y in test_dataset])
    y_pred_probs = model.predict(test_dataset)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Compute metrics
    accuracy = accuracy_score(y_true, y_pred)
    report_dict = {
        "classification_metrics": {
            "accuracy": {"value": accuracy},
        }
    }

    # Save evaluation results
    output_dir = "/opt/ml/processing/evaluation"
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)

    evaluation_path = f"{output_dir}/evaluation.json"
    with open(evaluation_path, "w") as f:
        json.dump(report_dict, f)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:21                                                                                   │
│                                                                                                  │
│   18 │   # extracted_model_path = "/opt/ml/processing/model"                                     │
│   19 │                                                                                           │
│   20 │   # Load model                                                                            │
│ ❱ 21 │   model = tf.keras.models.load_model(model_pathing)                                       │
│   22 │   # model = pickle.load(open("cnn-model", "rb"))                                          │
│   23 │                                                                                           │
│   24 │   # Load test images                                                                      │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/keras/src/saving/saving_api.py:196 in load_model         │
│                                                                                                  │
│   193 │   │   │   safe_mode=safe_mode,                                                           │
│   194 │   │   )                                                                                  │
│   195 │   if str(filepath).endswith((".h5", ".hdf5")):                                           │
│ ❱ 196 │   │   return legacy_h5_format.load_model_from_hdf5(                                      │
│   197 │   │   │   filepath, custom_objects=custom_objects, compile=compile                       │
│   198 │   │   )                                                                                  │
│   199 │   elif str(filepath).endswith(".keras"):                                                 │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/keras/src/legacy/saving/legacy_h5_format.py:116 in       │
│ load_model_from_hdf5                                                                             │
│                                                                                                  │
│   113 │                                                                                          │
│   114 │   opened_new_file = not isinstance(filepath, h5py.File)                                  │
│   115 │   if opened_new_file:                                                                    │
│ ❱ 116 │   │   f = h5py.File(filepath, mode="r")                                                  │
│   117 │   else:                                                                                  │
│   118 │   │   f = filepath                                                                       │
│   119                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/h5py/_hl/files.py:561 in __init__                        │
│                                                                                                  │
│   558 │   │   │   │   fcpl = make_fcpl(track_order=track_order, fs_strategy=fs_strategy,         │
│   559 │   │   │   │   │   │   │   │    fs_persist=fs_persist, fs_threshold=fs_threshold,         │
│   560 │   │   │   │   │   │   │   │    fs_page_size=fs_page_size)                                │
│ ❱ 561 │   │   │   │   fid = make_fid(name, mode, userblock_size, fapl, fcpl, swmr=swmr)          │
│   562 │   │   │                                                                                  │
│   563 │   │   │   if isinstance(libver, tuple):            

In [ ]:
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.processing import ProcessingInput, ProcessingOutput

processing_step = ProcessingStep(
    name="EvaluationStep",
    processor=processor,  
    inputs=[
        ProcessingInput(
            source=f"s3://{s3_bucket}/{s3_test_data_prefix}",
            destination="/opt/ml/processing/test"
        ),
        ProcessingInput(
            source=f"s3://{s3_bucket}/{s3_model_prefix}/model.tar.gz",
            destination="/opt/ml/processing/model"
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation")
    ],
    code="code/evaluation.py"
)


In [ ]:
# From example notebook, use if the top section isn't working well
from sagemaker.processing import ScriptProcessor


script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="script-fer-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="code/evaluation.py",
)

In [ ]:
from sagemaker.workflow.properties import PropertyFile


evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)
step_eval = ProcessingStep(
    name="FEREval",
    step_args=eval_args,
    property_files=[evaluation_report],
)

### Register Model
A model package is an abstraction of reusable model artifacts that packages all the ingredients required for inference. Primarily, it consistes of an inference specification that defines the inference image to use along with an option model weights location.

A model package group is a collection of model packages. A model package group can be created for a specific ML business problem, and new versions of the model packages can be added to it. Typically, customers are expected to create a ModelPackageGroup for a SageMaker pipeline so that model package versions can be added to the group for every SageMaker pipeline run.

In [ ]:
from sagemaker.model_metrics import MetricsSource, ModelMetrics

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)
step_register = ModelStep(name="FERRegisterModel", step_args=register_args)

### Define a Fail Step to Terminate the Pipeline 
When the threshold accuarcy value is not met, there needs to be failure handeling so the failed model is not published and the pipeline can exit gracefully. 

In [ ]:
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join

step_fail = FailStep(
    name="FERAccuracyFail",
    error_message=Join(on=" ", values=["Execution failed due to Accuracy <", accuracy_threshold]),
)

In [ ]:
# Accuracy condition met
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet


cond_lte = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="regression_metrics.accuracy.value",
    ),
    right=accuracy_threshold,
)

step_cond = ConditionStep(
    name="AbaloneMSECond",
    conditions=[cond_lte],
    if_steps=[step_register],
    else_steps=[step_fail],
)

### Define a pipeline of parameters, steps and conditions
In this section, the steps defined above are combined into a Pipeline so it can be executed.

A pipeline requires a `name`, `parameters`, and `steps`. 

In [ ]:
from sagemaker.workflow.pipeline import Pipeline


pipeline_name = f"AbalonePipeline"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        instance_type,
        model_approval_status,
        input_data,
        batch_data,
        mse_threshold,
    ],
    steps=[step_eval, step_cond],
)

In [ ]:
# Check pipeline definition
import json


definition = json.loads(pipeline.definition())
definition

### Submit pipeline to SageMaker and Execute
Submit the pipeline definition to the Pipeline service. The Pipeline service uses the role that is passed in to create all the jobs defined in the steps.

In [ ]:
pipeline.upsert(role_arn=role)


In [ ]:
# Start the pipeline and accept all the default parameters
execution = pipeline.start()

### Pipeline Operations: Examining and waiting for pipeline execution
Describe the pipeline and examing the execution

In [ ]:
execution.describe()

In [ ]:
execution.wait()

In [ ]:
# Shows which steps have been executed
execution.list_steps()

# Evaluate the execution
Examine the resulting model evaluation after the pipeline completes. Download the resulting `evaluation.json` file from s3 and print the report

In [ ]:
from pprint import pprint


evaluation_json = sagemaker.s3.S3Downloader.read_file(
    "{}/evaluation.json".format(
        step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
    )
)
pprint(json.loads(evaluation_json))

In [ ]:
# Review the lineage of the artifacts generated by the pipeline
import time
from sagemaker.lineage.visualizer import LineageTableVisualizer


viz = LineageTableVisualizer(sagemaker.session.Session())
for execution_step in reversed(execution.list_steps()):
    print(execution_step)
    display(viz.show(pipeline_execution_step=execution_step))
    time.sleep(5)

### Parameterized Executions
This section allows for different parameters to be adjusted when running the pipeline. This is useful given the inability for the team to share an S3 bucket, and potential other manipulations that need to occur in order to fully run the pipeline. This allows for adjustments as different models are loaded and tested as well.

In [ ]:
execution = pipeline.start(
    parameters=dict(
        s3_bucket="additional_value_to_test",
        s3_test_data_prefix="test",
        s3_model_prefix="models",
        accuracy_threshold="0.4"
    )
)